# DS1 — Feature Extraction

**Dataset:** Dataset 1 — Compact friction intervention testing  
**Stage:** 1 — Dataset-specific feature extraction (no cross-dataset harmonization)  
**Input files:** `outputs/cleaned/ds1_*.csv`  
**Output files:** `outputs/features/ds1/`

---

## Outputs produced

| File | Rows | Contents |
|---|---|---|
| `ds1_behavior_features.csv` | ~619 | All behavioral participants |
| `ds1_survey_features.csv` | ~572 | Participants with pre-survey + behavioral |
| `ds1_prepost_features.csv` | ~169 | Participants with behavioral + both surveys |

---

## Event definitions

- **What counts as an event row:** Every row in `ds1_behavioral_clean.csv` represents a
  single OneSec friction-intervention event (app open attempt by participant).
- **Intervention exposure:** All rows — every DS1 event implies friction was shown.
- **Outcome-bearing rows:** Rows where `event_outcome_simple` ∈ {opened, dismissed, closed}.
  Excluded: 467 rows where `event_outcome_simple == 'missing'` (0.1%).
- **Longitudinal axis:** `secondsSinceOneSecSetup` — available for 81.1% of events.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

CLEANED = PROJECT_ROOT / "outputs" / "cleaned"
OUT = PROJECT_ROOT / "outputs" / "features" / "ds1"
OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output folder:", OUT)

## 1. Load cleaned files

In [ ]:
from src.features.ds1_features import load_ds1_files

beh, pre, post, linkage = load_ds1_files(CLEANED)

print(f"Behavioral rows:      {len(beh):,}")
print(f"Behavioral participants: {beh['participant_id_canonical'].nunique():,}")
print(f"Pre-survey rows:      {len(pre):,}")
print(f"Post-survey rows:     {len(post):,}")
print(f"Linkage rows:         {len(linkage):,}")

In [ ]:
# Verify linkage overlap
beh_ids = set(beh['participant_id_canonical'].unique())
pre_ids = set(pre['participant_id_canonical'].unique())
post_ids = set(post['participant_id_canonical'].unique())

print(f"Behavioral only:            {len(beh_ids - pre_ids - post_ids):,}")
print(f"Behavioral + pre-survey:    {len(beh_ids & pre_ids):,}")
print(f"Behavioral + post-survey:   {len(beh_ids & post_ids):,}")
print(f"All three (fully linked):   {len(beh_ids & pre_ids & post_ids):,}")

## 2. Build behavior-only features

Feature groups computed here:
- **G1** Volume and activity (DIRECT, High)
- **G2** Outcome distribution (DIRECT, High)
- **G3** App structure (DIRECT, High)
- **G4** Temporal structure (DIRECT-DERIVED, High)
- **G5** Setup-time window (DIRECT, High)
- **G6** Longitudinal adaptation via setup-time (PROXY, Medium)
- **G7** Override tendency streaks (PROXY, Medium)

In [ ]:
from src.features.ds1_features import build_ds1_behavior_features

print("Building behavioral features...")
beh_feats = build_ds1_behavior_features(beh)
print(f"Shape: {beh_feats.shape}")
beh_feats.head(3)

In [ ]:
# Validation: check for expected row count and no duplicate participant IDs
assert beh_feats['participant_id_canonical'].nunique() == len(beh_feats), \
    "Duplicate participant IDs in output"
print(f"Unique participants: {len(beh_feats):,}")
print("\nMissingness summary (selected features):")
cols_to_check = [
    'total_events', 'rate_opened', 'rate_dismissed', 'rate_closed',
    'app_entropy', 'setup_window_days', 'open_rate_change_setup',
    'max_consecutive_opens'
]
print(beh_feats[cols_to_check].isna().sum())

In [ ]:
# Distribution summary for key features
print(beh_feats[['total_events', 'obs_window_days', 'events_per_day',
                  'rate_opened', 'rate_dismissed', 'rate_closed',
                  'n_unique_apps', 'app_entropy']].describe().round(3))

In [ ]:
# Adaptation feature coverage
# NOTE: n_events_with_setup_time is a COUNT (0 when absent), never NaN — test with > 0.
n_setup = beh_feats['n_events_with_setup_time']
n_eligible = (n_setup > 0).sum()
n_with_adaptation = beh_feats['open_rate_change_setup'].notna().sum()
print(f"Participants with any setup-time data: {n_eligible:,} / {len(beh_feats):,}")
print(f"  0 setup-time rows:   {(n_setup == 0).sum():,}")
print(f"  1-4 setup-time rows: {((n_setup >= 1) & (n_setup <= 4)).sum():,}  (NaN setup_window_days, <5 gate)")
print(f"  >=5 setup-time rows: {(n_setup >= 5).sum():,}")
print(f"Participants with adaptation features: {n_with_adaptation:,}")

In [ ]:
# Save
out_path = OUT / "ds1_behavior_features.csv"
beh_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(beh_feats):,} rows, {beh_feats.shape[1]} columns)")

## 3. Build pre-survey features

Feature groups:
- **G7** Pre-survey psychosocial baseline (DIRECT, High)  
  Available for participants with behavioral + pre-survey data (~572 expected)

In [ ]:
from src.features.ds1_features import build_ds1_survey_features

survey_feats = build_ds1_survey_features(beh, pre, linkage)
print(f"Shape: {survey_feats.shape}")
print("\nMissingness summary:")
print(survey_feats[['problematic_cons_pre', 'happy_pre', 'fomo_mean_pre' if 'fomo_mean_pre' in survey_feats.columns else 'n_problems_selected']].isna().sum())
survey_feats.head(3)

In [ ]:
out_path = OUT / "ds1_survey_features.csv"
survey_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(survey_feats):,} rows, {survey_feats.shape[1]} columns)")

## 4. Build pre-post change features

Feature groups:
- **G8** Post-survey (DIRECT, High)  
- **G9** Pre-post signed change (DIRECT-DERIVED, Medium)  
Available for: behavioral + pre-survey + post-survey (~169 expected)

In [ ]:
from src.features.ds1_features import build_ds1_prepost_features

prepost_feats = build_ds1_prepost_features(beh, pre, post)
print(f"Shape: {prepost_feats.shape}")
print("\nChange feature summary:")
change_cols = [c for c in prepost_feats.columns if c.endswith('_change')]
print(prepost_feats[change_cols].describe().round(3))

In [ ]:
out_path = OUT / "ds1_prepost_features.csv"
prepost_feats.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(prepost_feats):,} rows, {prepost_feats.shape[1]} columns)")

## 5. Output summary

In [ ]:
print("=" * 60)
print("DS1 Feature Extraction — Output Summary")
print("=" * 60)

for fname, df in [
    ('ds1_behavior_features.csv', beh_feats),
    ('ds1_survey_features.csv', survey_feats),
    ('ds1_prepost_features.csv', prepost_feats),
]:
    n_features = df.shape[1] - 1  # exclude participant ID
    print(f"  {fname}")
    print(f"    {len(df):,} participants × {n_features} features")
    missing_any = df.drop(columns='participant_id_canonical').isna().any(axis=1).sum()
    print(f"    Participants with any NaN feature: {missing_any:,}")
    print()